*Run this notebook on your own computer* — in VS Code, or any editor that opens `.ipynb` files. If a lab needs a GPU (mainly Sessions 9–12), open it in VS Code and choose **Select Kernel → Colab** to run it on a Colab machine without leaving the editor; see [Setup](https://course.bioinfo.se/setup.html).

# Session 12 lab: AlphaFold in practice

Most of this lab happens in web tools: the **AlphaFold Server**
(alphafoldserver.com, AlphaFold3), the **AlphaFold Protein Structure
Database** (alphafold.ebi.ac.uk) and PyMOL. This notebook is the helper:

- **Part A** fetches the chain sequences of four target complexes from the
  PDB, ready to paste into the AlphaFold Server. There is one target per
  "region" of the lecture's framework for where structure prediction
  works and where it does not, and you run all four.
- **Part B** reads AlphaFold DB models: per-residue confidence (pLDDT),
  the predicted aligned error (PAE), disordered regions, and a comparison
  with an experimental structure.
- **Part C** reads the results `.zip` that the AlphaFold Server gives you:
  pTM, ipTM, the PAE heatmap, and the interface score **ipSAE**.
- **Part D** puts your four results side by side.

Every `todo("...")` call marks a piece of code for you to write: replace
the whole `todo(...)` call with your code. Work through the cells **in
order**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

**The four targets** (one per region):

| Region | What makes it hard | Target | PDB |
|---|---|---|---|
| I, tractable | nothing: stable interface, strong co-evolution | Ephrin-A5 : EphB2 | 1SHW |
| II, ambiguity and competition | a plausible interface is easy; the *right partner* is hard | Izumo1 : Juno | 5F4E |
| III, missing co-evolution | no joint evolutionary pressure between the partners | antibody : antigen | 8TQ7 |
| IV, missing context and dynamics | disorder-to-order folding on binding | p53 transactivation peptide : MDM2 | 1YCR |

Each group runs the target of the region it has been assigned. The
AlphaFold Server allows 20 jobs per day and account, which is plenty for
one target.


In [ ]:
# Install the libraries Colab does not come with (does nothing if they are already installed)
import importlib.util, subprocess, sys
for module, package in [("Bio", "biopython")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [ ]:
import os, io, re, json, glob, zipfile, tempfile, subprocess, sys
import numpy as np
import requests
import matplotlib.pyplot as plt

WORK = tempfile.mkdtemp(prefix="kb8029_session12_")      # all downloads go here, not next to the notebook
print("working directory for downloads:", WORK)

try:
    import tmtools
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tmtools"], check=True)
    import tmtools
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import three_to_index, index_to_one

def todo(what):
    '''Placeholder for code you write: replace the whole todo(...) call with your own code.'''
    raise NotImplementedError(f"TODO in this cell: {what}")

## Part A: get the sequences

The RCSB PDB serves every entry's sequences as FASTA. Each record is one
*entity* (a distinct chain type). Its header lists which chains in the
file carry that sequence, for example `Chains A, C`. For the AlphaFold
Server, add one **Protein** entity per distinct sequence, with *copies*
set to 1 (we predict one copy of each complex).

In [ ]:
TARGETS = {"I": ("Ephrin-A5 : EphB2", "1SHW"), "II": ("Izumo1 : Juno", "5F4E"),
           "III": ("antibody : antigen", "8TQ7"), "IV": ("p53 TAD : MDM2", "1YCR")}

def pdb_entities(pdb_id):
    '''Return a list of (header, sequence) pairs, one per entity of a PDB entry.'''
    text = requests.get(f"https://www.rcsb.org/fasta/entry/{pdb_id}", timeout=60).text
    records = []
    for block in text.strip().split(">")[1:]:
        header, *seq_lines = block.strip().split("\n")
        records.append((header, "".join(seq_lines)))
    return records

entities = {}
for region, (name, pdb_id) in TARGETS.items():
    entities[region] = pdb_entities(pdb_id)
    print(f"Region {region}: {name} ({pdb_id})")
    for header, seq in entities[region]:
        n_res = todo("the number of residues in this entity's sequence")
        print(f"   {header[:90]}   [{n_res} residues]")

In [ ]:
print("Paste these into the AlphaFold Server, one Protein entity each (copies = 1).")
print("Four separate jobs, one per region. The cap is 20 jobs a day, so all four fit.\n")
for region, (name, pdb) in TARGETS.items():
    print("=" * 70)
    print(f"JOB: region {region} -- {name} ({pdb})")
    print("=" * 70)
    for header, seq in entities[region]:
        print(f">{header}\n{seq}\n")


## Part B: reading AlphaFold DB models

The AlphaFold DB API returns, for a UniProt accession, links to the model
(its B-factor column holds **pLDDT**, 0-100) and to the **PAE** matrix.
pLDDT > 90 is very high confidence, 70-90 confident, 50-70 low, and below
50 very low. Very low confidence is not a prediction of disorder:
disorder is one common reason, but too few homologs, several conformations
or an uncertain domain orientation give the same low score.

In [ ]:
def afdb_entry(accession):
    '''Metadata, parsed structure and per-residue pLDDT of the AlphaFold DB model for a UniProt accession.'''
    meta = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{accession}", timeout=60).json()[0]
    pdb_text = requests.get(meta["pdbUrl"], timeout=60).text
    structure = PDBParser(QUIET=True).get_structure(accession, io.StringIO(pdb_text))
    plddt = np.array([res["CA"].get_bfactor() for res in structure.get_residues()])
    return meta, structure, plddt

meta, s_p94485, plddt = afdb_entry("P94485")
print(meta["uniprotDescription"], "|", len(plddt), "residues | model version", meta["latestVersion"])
pae = np.array(requests.get(meta["paeDocUrl"], timeout=60).json()[0]["predicted_aligned_error"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(np.arange(1, len(plddt) + 1), plddt)
for y, c in [(90, "#2f6db5"), (70, "#7fb3e6"), (50, "#f0a030")]:
    a1.axhline(y, color=c, ls="--", lw=0.8)
a1.set_xlabel("residue"); a1.set_ylabel("pLDDT"); a1.set_ylim(0, 100); a1.set_title("P94485: pLDDT per residue")
im = a2.imshow(pae, cmap="Greens_r", vmin=0, vmax=30)
a2.set_xlabel("aligned on residue"); a2.set_ylabel("error of residue"); a2.set_title("P94485: PAE (Å)")
plt.colorbar(im, ax=a2, fraction=0.046); plt.tight_layout(); plt.show()

**Disorder.** Which of these four models is the most disordered? Compare
the fraction of residues with very low confidence (pLDDT < 50), which the
AlphaFold DB API also reports directly.

In [ ]:
for acc in ["A8MVM7", "Q96NG5", "Q8IYP9", "Q8W3K0"]:
    m, _, p = afdb_entry(acc)
    frac_low = todo('fraction of residues with pLDDT below 50')
    print(f"{acc}: {m['uniprotDescription'][:45]:45s} {len(p):5d} residues, mean pLDDT {p.mean():5.1f}, "
          f"pLDDT < 50: {frac_low:.2f}  (API: {m['fractionPlddtVeryLow']:.2f})")

**Model vs. experiment.** The AlphaFold DB model of mouse Ephrin-A5
(UniProt O08543) against the crystal structure of the same domain in
1SHW, chain A. TM-align superimposes the two and reports the RMSD over
the aligned Cα atoms and the TM-score (0 to 1; above about 0.5 means the
same fold).

In [ ]:
def ca_chain(structure, chain_id=None):
    chain = structure[0][chain_id] if chain_id else next(iter(structure[0]))
    residues = [r for r in chain if r.id[0] == " " and "CA" in r]
    seq = "".join(index_to_one(three_to_index(r.get_resname())) for r in residues)
    return np.array([r["CA"].coord for r in residues]), seq

_, s_model, _ = afdb_entry("O08543")
crystal = PDBParser(QUIET=True).get_structure("1SHW", io.StringIO(requests.get("https://files.rcsb.org/download/1SHW.pdb", timeout=60).text))
X_model, seq_model = ca_chain(s_model)
X_xtal, seq_xtal = ca_chain(crystal, "A")
result = tmtools.tm_align(X_model, X_xtal, seq_model, seq_xtal)
print(f"model {len(seq_model)} residues, crystal chain A {len(seq_xtal)} residues")
print(f"TM-score (normalised by the crystal chain) {result.tm_norm_chain2:.3f}, RMSD {result.rmsd:.2f} Å")

## Part C: your four AlphaFold Server results

Download the result of each job from the AlphaFold Server (the `.zip`
button) and put the four files where this notebook can reach them. In
Colab, use the folder icon in the sidebar to upload them; they land in
`/content`, so the path of each is `/content/<name of the zip>`. In VS
Code, Command Palette → **Colab: Upload** does the same thing.

Each zip holds five models (`model_0` is the top-ranked), and for each a
`summary_confidences` file (pTM, ipTM, chain-pair ipTM) and a `full_data`
file (the PAE matrix and per-token chain IDs).

**How long this takes.** The Server runs your four jobs in parallel, so
you wait for the slowest, not for four in a row. Measured on 24 November
2026: Regions I and IV came back in a minute, Region II in two, and
Region III — five chains and 810 residues — in six. Submit all four at
the end of Part A and work on Part B while they run.

**If a job is still running after about twenty minutes, or the Server is
down,** leave that region's entry empty below: the cell falls back to a
precomputed result from the course repository. Say in your answers which
regions you fell back on — a fallback is not a failed lab, but it is
part of the result.


In [ ]:
# The zip you downloaded for each region. Leave one empty to use the
# precomputed result from the course repository instead.
MY_ZIPS = {
    "I":   "",   # Ephrin-A5 : EphB2   (1SHW)
    "II":  "",   # Izumo1 : Juno       (5F4E)
    "III": "",   # antibody : antigen  (8TQ7)
    "IV":  "",   # p53 TAD : MDM2      (1YCR)
}
# The precomputed results, stored in the public course repository as ordinary
# files. The media host is tried second in case they are ever moved to Git LFS,
# because raw.githubusercontent.com serves an LFS file as its pointer, not as the
# file; the PK check below is what tells the two apart.
REPO = "ElofssonLab/kb8029-book/main/notebooks/data/session12-af3-examples"
BACKUPS = [f"https://raw.githubusercontent.com/{REPO}",
           f"https://media.githubusercontent.com/media/{REPO}"]

def fetch_backup(pdb):
    """The precomputed zip for one target, as bytes, or None."""
    for base in BACKUPS:
        r = requests.get(f"{base}/{pdb}.zip", timeout=300)
        if r.status_code == 200 and r.content[:2] == b"PK":   # a real zip, not a pointer
            return r.content
    return None

paths, fell_back = {}, []
for region, (name, pdb) in TARGETS.items():
    zip_path = MY_ZIPS.get(region, "").strip()
    if not zip_path:
        blob = fetch_backup(pdb)
        if blob is None:
            raise SystemExit(f"Region {region}: no zip of your own, and no precomputed "
                             f"result for {pdb} could be downloaded; ask a teacher.")
        zip_path = os.path.join(WORK, f"{pdb}.zip")
        open(zip_path, "wb").write(blob)
        fell_back.append(region)
    job_dir = os.path.join(WORK, f"job_region_{region}")
    zipfile.ZipFile(zip_path).extractall(job_dir)
    paths[region] = {
        "summary": sorted(glob.glob(os.path.join(job_dir, "*summary_confidences_0.json")))[0],
        "full":    sorted(glob.glob(os.path.join(job_dir, "*full_data_0.json")))[0],
        "cif":     sorted(glob.glob(os.path.join(job_dir, "*model_0.cif")))[0],
    }
    mark = "  (precomputed)" if region in fell_back else ""
    print(f"Region {region:3s} {name:24s} {os.path.basename(paths[region]['cif'])}{mark}")
if fell_back:
    print(f"\nFell back to precomputed results for: {', '.join(fell_back)}. Say so in your answers.")


In [ ]:
def ptm_and_iptm(summary):
    '''Pull pTM and ipTM out of one summary_confidences dictionary.'''
    ptm = todo('the pTM value from the summary dictionary')
    iptm = todo('the ipTM value from the summary dictionary')
    return float(ptm), float(iptm)

scores = {}
print(f"{'region':7s} {'target':24s} {'pTM':>5s} {'ipTM':>6s} {'rank':>6s}")
for region, (name, pdb) in TARGETS.items():
    summary = json.load(open(paths[region]["summary"]))
    ptm, iptm = ptm_and_iptm(summary)
    scores[region] = {"ptm": ptm, "iptm": iptm,
                      "chain_pair_iptm": summary["chain_pair_iptm"],
                      "ranking_score": summary["ranking_score"]}
    print(f"{region:7s} {name[:24]:24s} {ptm:5.2f} {iptm:6.2f} {summary['ranking_score']:6.2f}")

print("\nchain-pair ipTM:")
for region in TARGETS:
    print(f"  region {region}")
    for row in scores[region]["chain_pair_iptm"]:
        print("     ", row)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.5, 9.5))
for ax, (region, (name, pdb)) in zip(axes.ravel(), TARGETS.items()):
    full = json.load(open(paths[region]["full"]))
    pae_af3 = np.array(full["pae"])
    chains = np.array(full["token_chain_ids"])
    boundaries = [i for i in range(1, len(chains)) if chains[i] != chains[i - 1]]
    im = ax.imshow(pae_af3, cmap="Greens_r", vmin=0, vmax=30)
    for b in boundaries:
        ax.axhline(b - 0.5, color="red", lw=0.8); ax.axvline(b - 0.5, color="red", lw=0.8)
    ax.set_title(f"Region {region}: {name} ({pdb})", fontsize=9)
    ax.set_xlabel("aligned on token"); ax.set_ylabel("error of token")
    print(f"region {region} tokens per chain:",
          {c: int((chains == c).sum()) for c in dict.fromkeys(chains)})
fig.colorbar(im, ax=axes, label="PAE (Å)", fraction=0.03)
fig.suptitle("PAE of each top-ranked model; red lines = chain boundaries", fontsize=11)
plt.show()


**ipSAE.** ipTM is computed over *all* residue pairs between two chains,
so long disordered tails or non-interacting domains pull it down (or add
noise) even when the interface itself is modelled well. **ipSAE**
(Dunbrack, 2025) uses only residue pairs whose PAE is below a cutoff.
The script is downloaded from its GitHub repository and run on the
top-ranked model with a PAE cutoff and a distance cutoff of 10 Å each.

In [ ]:
ipsae_script = os.path.join(WORK, "ipsae.py")
if not os.path.exists(ipsae_script):
    open(ipsae_script, "w").write(requests.get(
        "https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py", timeout=60).text)

for region, (name, pdb) in TARGETS.items():
    full_file, cif_file = paths[region]["full"], paths[region]["cif"]
    subprocess.run([sys.executable, ipsae_script, full_file, cif_file, "10", "10"],
                   check=True, capture_output=True)
    out_file = cif_file.replace(".cif", "_10_10.txt")
    rows = [line.split() for line in open(out_file) if line.strip()]
    header, rows = rows[0], rows[1:]
    col = {nm: k for k, nm in enumerate(header)}
    print(f"\nregion {region}: {name} ({pdb})")
    print(f"  {'chains':8s} {'type':5s} {'ipSAE':>7s} {'ipTM_af':>8s} {'pDockQ':>7s}")
    for r in rows:
        if len(r) > col["ipSAE"]:
            print(f"  {r[col['Chn1']]}-{r[col['Chn2']]:5s} {r[col['Type']]:5s} "
                  f"{float(r[col['ipSAE']]):7.3f} {float(r[col['ipTM_af']]):8.3f} "
                  f"{float(r[col['pDockQ']]):7.3f}")
    scores[region]["ipsae"] = max(float(r[col["ipSAE"]]) for r in rows
                                  if len(r) > col["ipSAE"] and r[col["Type"]] == "max")
    print(f"  ipSAE (max over chain pairs): {scores[region]['ipsae']:.3f}")


## Part D: the four regions side by side

You ran all four targets, so the comparison is yours to make and the
numbers are already in `scores` from Parts C. Plot ipTM against ipSAE for
each region, then ask the question the lecture asks: does the model\'s
confidence track how hard the problem really is?


In [ ]:
regions = list(TARGETS)
x = np.arange(len(regions))
plt.figure(figsize=(6.5, 3.8))
plt.bar(x - 0.2, [scores[r]["iptm"] for r in regions], width=0.4, label="ipTM")
plt.bar(x + 0.2, [scores[r]["ipsae"] for r in regions], width=0.4, label="ipSAE")
plt.xticks(x, [f"Region {r}" for r in regions]); plt.ylim(0, 1); plt.legend()
plt.ylabel("interface confidence")
plt.title("Interface confidence by region"); plt.show()

print(f"{'region':7s} {'target':24s} {'pTM':>5s} {'ipTM':>6s} {'ipSAE':>6s}")
for r in regions:
    print(f"{r:7s} {TARGETS[r][0][:24]:24s} {scores[r]['ptm']:5.2f} "
          f"{scores[r]['iptm']:6.2f} {scores[r]['ipsae']:6.2f}")


## Hand in

Upload this notebook (File → Download → .ipynb) and your PyMOL images
with the lab quiz on Canvas.